In [1]:
import numpy as np
import pandas as pd
import time
import math

### PYTHON ###

# Calculate the value of Pi using the Monte Carlo method
def calculate_pi(samples: list) -> pd.DataFrame:
    results = []

    for sample in samples:
        start_time = time.perf_counter()
        inside_circle = 0

        for _ in range(sample):
            x, y = np.random.uniform(0, 1, 2)
            if x**2 + y**2 <= 1:
                inside_circle += 1

        pi_estimate = (inside_circle / sample) * 4
        elapsed_time = time.perf_counter() - start_time
        diff = abs(pi_estimate - math.pi)

        results.append({
            "epochs": sample,
            "pi_estimate": pi_estimate,
            "difference": diff,
            "time_seconds": elapsed_time
        })

    return pd.DataFrame(results)

samples = [10, 100, 1000, 10000, 100000, 1000000, 10000000]
dataframe = calculate_pi(samples)
print(dataframe)

     epochs  pi_estimate  difference  time_seconds
0        10     1.600000    1.541593      0.000079
1       100     3.160000    0.018407      0.000196
2      1000     3.132000    0.009593      0.001912
3     10000     3.132800    0.008793      0.017734
4    100000     3.138560    0.003033      0.178993
5   1000000     3.143960    0.002367      1.756811
6  10000000     3.141219    0.000374     17.197642


In [ ]:
### PYTHON és més lent perquè ho fa tot seguit en un sol nucli, milions d'iteracions. Per això satura el nucli i va més lent

In [2]:
from pyspark.sql import SparkSession
import random

### PYSPARK ###

def calc_pi_rdd(sc, N, partitions=None):
    if partitions is None:
        partitions = sc.defaultParallelism

    start = time.time()

    def montecarlo_partition(iterator):
        rng = random.Random()

        inside = []
        for _ in iterator:
            x = rng.random()
            y = rng.random()
            inside.append(x*x + y*y <= 1)

        return inside

    rdd = sc.range(0, N, numSlices=partitions)

    inside_total = rdd.mapPartitions(montecarlo_partition).sum()
    pi = inside_total / N * 4
    return pi, time.time() - start

def build_dataframe(samples):
    results = []
    for sample in samples:
        appr, tim = calc_pi_rdd(sc, sample)
        results.append({
            'samples': sample,
            'approximation': appr,
            'error': abs(math.pi - appr),
            'time': tim
        })

    return pd.DataFrame(results)

spark = SparkSession.builder.getOrCreate()
sc = spark.sparkContext

samples = [10, 100, 1000, 10000, 100000, 1000000, 10000000, 100000000]

data = build_dataframe(samples)

print(data)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/27 19:54:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
[Stage 7:>                                                          (0 + 4) / 4]

     samples  approximation     error      time
0         10       3.600000  0.458407  0.969465
1        100       3.000000  0.141593  0.234380
2       1000       3.072000  0.069593  0.112819
3      10000       3.132800  0.008793  0.098136
4     100000       3.150760  0.009167  0.097795
5    1000000       3.140568  0.001025  0.159207
6   10000000       3.140915  0.000677  0.444446
7  100000000       3.141172  0.000420  2.102321


## Quina diferència de temps heu notat?
Es nota una diferència de temps significativa. Python passa dels 20 segons, arribant a 221 segons a 100000000 iteracions, mentres que PySpark triga 2 segons en fer aquestes mateixes iteracions.

## Què heu observat a la Spark UI (Tasks vs Particions)?
Cada task està formada per diverses partitions, que distribueixen i processen les dades en fragments i s'ajunten totes al final.

## Per què creieu que RDD és millor per aquest cas que un DataFrame?
Particionar tasques i paral·lelitzar els processos fa que les execucions siguin molt més ràpides. Amb poques iteracions no val la pena, però quan pugen, l'avantatge de temps és exponencial.